# 04 — Tree pipeline attribution

Fit on **train**; permutation importance on **selection** (2022–2023) only — not final 2024.


In [ ]:
from spy_alpha_research.data import (
    load_config, project_root, load_frame, save_frame, split_three_way, make_target,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]

from spy_alpha_research.pipelines import fit_random_forest, fit_hist_gbm
from spy_alpha_research.reporting import plot_importance

X_train = load_frame(PROC / "features_train_scaled.parquet")
X_sel = load_frame(PROC / "features_selection_scaled.parquet")
y = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
rf = fit_random_forest(
    X_train, y.reindex(X_train.index), X_sel, y.reindex(X_sel.index),
    random_state=exp["project"]["seed"], **models["trees"]["random_forest"],
)
hgb = fit_hist_gbm(
    X_train, y.reindex(X_train.index), X_sel, y.reindex(X_sel.index),
    random_state=exp["project"]["seed"], **models["trees"]["hist_gbm"],
)
plot_importance(rf.importances, FIG / "tree_importance.png", title="RF perm-importance (selection)")
rf.importances.head(10), hgb.importances.head(10)
